# PRE REQ

In [555]:
import pandas as pd
import random
from collections import defaultdict, Counter

In [556]:
POPULASI = 500
VIOLATION_COST = 100
ITERATION = 100
MUTATION_PROB = 0.8
CROSSOVER_PROB = 0.7 
TOURNAMENT_SIZE = 20
SLOT_PER_KELAS = 36
JUMLAH_KELAS = 27
ELITE_SIZE = 5

In [557]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

# DICT

In [558]:
# =========================================================
# Mapping hari ke id
# =========================================================
hariId = {
    "Senin": 1,
    "Selasa": 2,
    "Rabu": 3,
    "Kamis": 4,
    "Jumat": 5,
}

# =========================================================
# Slot per hari
# =========================================================
slotPerHari = slot_df['hari'].value_counts().to_dict()
slotPerHari = {hariId[k]: v for k, v in slotPerHari.items()}

# =========================================================
# Guru ID -> Nama Guru
# =========================================================
guruPengajar = dict(
    zip(guru_df['guru_id'], guru_df['nama_guru'])
)

# =========================================================
# Kelas -> Tingkatan
# (lebih efisien dibanding nested dictionary sebelumnya)
# =========================================================
kelasTingkatan = dict(
    zip(kelas_df['kelas_id'], kelas_df['tingkatan'])
)

# daftar kelas yang sudah disortir
kelasIds = sorted(kelasTingkatan.keys())

# =========================================================
# Kelas -> Nama Kelas
# =========================================================
namaKelas = dict(
    zip(kelas_df['kelas_id'], kelas_df['nama_kelas'])
)

# =========================================================
# Mapel ID -> Nama Mapel
# =========================================================
namaMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['nama_mapel'])
)

# =========================================================
# Mapel -> Jam per minggu
# =========================================================
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)

# =========================================================
# Mapel -> Hari MGMP
# =========================================================
mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)

mgmpMapel = {
    mapel_id: hariId[hari]
    for mapel_id, hari in mgmpMapel.items()
}

# =========================================================
# Batas jam siang
# =========================================================
batasSiang = {
    1: 5,
    2: 5,
    3: 4,
    4: 5,
    5: 4
}

# =========================================================
# Batas MGMP
# =========================================================
batasMGMP = {
    1: 2,
    2: 2,
    3: 2,
    4: 2,
    5: 1
}

# =========================================================
# Guru -> mapel yang diajarkan
# =========================================================
durasiGuruMengajar = defaultdict(list)

for row in relasi_guru_mapel_df.itertuples():
    durasiGuruMengajar[row.guru_id].append({
        "mapel_id": row.mapel_id,
        "tingkatan": row.tingkatan,
        "durasi": row.durasi
    })

durasiGuruMengajar = dict(durasiGuruMengajar)

# =========================================================
# (OPTIMASI PENTING)
# Mapel + Tingkatan -> Guru Valid
# untuk mempercepat generate individu
# =========================================================
guruValidMapel = defaultdict(list)

for row in relasi_guru_mapel_df.itertuples():

    key = (row.mapel_id, row.tingkatan)

    guruValidMapel[key].append(row.guru_id)

guruValidMapel = dict(guruValidMapel)

# =========================================================
# Wali kelas
# =========================================================
waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)

# =========================================================
# Precompute daftar mapel
# untuk mempercepat looping
# =========================================================
mapelItems = list(jamPerMingguMapel.items())

# =========================================================
# Mapping kelas -> index kromosom
# =========================================================
kelasIndex = {}

slotRange = 0

for kelas_id in range(1, JUMLAH_KELAS + 1):

    end = slotRange + SLOT_PER_KELAS

    kelasIndex[kelas_id] = (slotRange, end)

    slotRange = end

# =========================================================
# Total slot kromosom
# =========================================================
TOTAL_SLOT = JUMLAH_KELAS * SLOT_PER_KELAS

# =========================================================
# Precompute slot -> hari
# (dipakai untuk MGMP dan constraint harian)
# =========================================================
slotHari = []

for row in slot_df.itertuples():

    slotHari.append(hariId[row.hari])

# =========================================================
# Index slot global
# contoh:
# index 0 = kelas1 slot1
# index 36 = kelas2 slot1
# =========================================================
slotGlobal = []

for kelas_id in range(1, JUMLAH_KELAS + 1):

    start, end = kelasIndex[kelas_id]

    for slot in range(start, end):

        slotGlobal.append({
            "kelas_id": kelas_id,
            "slot_kelas": slot - start
        })

# INDIVIDU

In [559]:
def blokDistribusi(jam):

    if jam == 2:
        return [2]

    if jam == 3:
        return [3]

    if jam == 4:
        return [2,2]

    if jam == 5:
        return [2,3]

    return [jam]

In [560]:
blokMapel = {}

for mapel_id, jam in jamPerMingguMapel.items():

    blokMapel[mapel_id] = blokDistribusi(jam)

In [561]:
def generatePerKelas(tingkatan):

    pilihan = []

    for mapel_id, jam in mapelItems:

        guruValid = guruValidMapel.get((mapel_id, tingkatan))

        if not guruValid:
            continue

        guru = random.choice(guruValid)

        for durasi in blokMapel[mapel_id]:

            pilihan.extend([(mapel_id, guru)] * durasi)

    random.shuffle(pilihan)

    # pastikan jumlah slot tepat
    if len(pilihan) > SLOT_PER_KELAS:
        pilihan = pilihan[:SLOT_PER_KELAS]

    if len(pilihan) < SLOT_PER_KELAS:

        while len(pilihan) < SLOT_PER_KELAS:

            mapel_id, jam = random.choice(mapelItems)

            guruValid = guruValidMapel.get((mapel_id, tingkatan))

            if guruValid:
                guru = random.choice(guruValid)
                pilihan.append((mapel_id, guru))

    return pilihan

In [562]:
def individuConstruct():

    individu = []

    for kelas_id in kelasIds:

        tingkatan = kelasTingkatan[kelas_id]

        jadwalKelas = generatePerKelas(tingkatan)

        individu.extend(jadwalKelas)

    return individu

In [563]:
def generatePopulation(pop_size):

    populasi = []

    for _ in range(pop_size):

        individu = individuConstruct()

        populasi.append(individu)

    return populasi

In [564]:
populasi = generatePopulation(POPULASI)

In [565]:
populasi[1]

[(6, 44),
 (3, 36),
 (11, 43),
 (5, 27),
 (3, 36),
 (3, 36),
 (4, 19),
 (12, 51),
 (1, 39),
 (2, 33),
 (13, 53),
 (1, 39),
 (7, 37),
 (5, 27),
 (6, 44),
 (7, 37),
 (10, 46),
 (9, 42),
 (3, 36),
 (5, 27),
 (6, 44),
 (10, 46),
 (4, 19),
 (8, 5),
 (9, 42),
 (7, 37),
 (12, 51),
 (5, 27),
 (13, 53),
 (8, 5),
 (4, 19),
 (12, 51),
 (11, 43),
 (3, 36),
 (4, 19),
 (2, 33),
 (5, 48),
 (12, 51),
 (13, 53),
 (8, 17),
 (7, 21),
 (10, 28),
 (2, 33),
 (5, 48),
 (5, 48),
 (12, 51),
 (7, 21),
 (7, 21),
 (6, 44),
 (8, 17),
 (10, 28),
 (2, 33),
 (4, 47),
 (3, 22),
 (4, 47),
 (11, 43),
 (1, 39),
 (9, 38),
 (12, 51),
 (4, 47),
 (6, 44),
 (3, 22),
 (4, 47),
 (11, 43),
 (5, 48),
 (1, 39),
 (13, 53),
 (6, 44),
 (3, 22),
 (3, 22),
 (9, 38),
 (3, 22),
 (3, 22),
 (3, 22),
 (11, 43),
 (9, 42),
 (2, 33),
 (10, 16),
 (4, 47),
 (6, 44),
 (12, 55),
 (11, 43),
 (3, 22),
 (6, 44),
 (7, 21),
 (7, 21),
 (1, 31),
 (2, 33),
 (12, 55),
 (5, 27),
 (12, 55),
 (5, 27),
 (9, 42),
 (7, 21),
 (8, 5),
 (4, 47),
 (10, 16),
 (1, 31)

# EVAL

In [566]:
slotAwalHari = {}
slotKeHari = {}

index = 0

for hari, jumlah in slotPerHari.items():

    slotAwalHari[hari] = index

    for _ in range(jumlah):

        slotKeHari[index] = hari
        index += 1

slotAkhirHari = {
1: 8,   # Senin
2: 16,  # Selasa
3: 24,  # Rabu
4: 31,  # Kamis
5: 36   # Jumat
}

In [567]:
def guruBentrok(individu):

    pelanggaran = 0

    for slot in range(SLOT_PER_KELAS):

        guruMengajar = []

        for kelas in range(JUMLAH_KELAS):

            index = kelas * SLOT_PER_KELAS + slot

            guruMengajar.append(individu[index][1])

        if len(guruMengajar) != len(set(guruMengajar)):

            pelanggaran += 1

    return pelanggaran

In [568]:
def durasiGuru(individu):

    loadGuru = Counter(guru for _, guru in individu)

    pelanggaran = 0

    for guru, count in loadGuru.items():

        if count > 40:

            pelanggaran += count - 40

    return pelanggaran

In [569]:
def distribusiMapel(individu):

    pelanggaran = 0

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for hari in range(1, 6):

            awal = start + slotAwalHari[hari]
            akhir = start + slotAkhirHari[hari]

            prev = None
            seen = set()

            for i in range(awal, akhir):

                mapel, guru = individu[i]

                if mapel != prev:

                    if mapel in seen:
                        pelanggaran += 1

                    seen.add(mapel)

                prev = mapel

    return pelanggaran

In [570]:
def mapelSiang(individu):

    pelanggaran = 0
    TARGET_MAPEL = 8

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for slot in range(SLOT_PER_KELAS):

            index = start + slot

            mapel_id = individu[index][0]

            if mapel_id != TARGET_MAPEL:
                continue

            hari = slotKeHari[slot]

            batas = batasSiang[hari]

            slotHari = slot - slotAwalHari[hari]

            if slotHari >= batas:

                pelanggaran += 1

    return pelanggaran

In [571]:
def waktuMGMP(individu):

    pelanggaran = 0

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for slot in range(SLOT_PER_KELAS):

            index = start + slot

            mapel_id = individu[index][0]

            hari = slotKeHari[slot]

            if mapel_id in mgmpMapel:

                if hari == mgmpMapel[mapel_id]:

                    pelanggaran += 1

    return pelanggaran

In [572]:
def cekWaliKelas(individu):

    pelanggaran = 0

    for guru_id, kelas_id in waliKelas.items():

        start = (kelas_id - 1) * SLOT_PER_KELAS
        end = start + SLOT_PER_KELAS

        kelasSlot = individu[start:end]

        mengajar = False

        for _, guru in kelasSlot:

            if guru == guru_id:

                mengajar = True
                break

        if not mengajar:

            pelanggaran += 1

    return pelanggaran

In [573]:
def konsistensiGuruMapel(individu):

    pelanggaran = 0

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS
        end = start + SLOT_PER_KELAS

        kelasSlot = individu[start:end]

        mapelGuru = {}

        for mapel, guru in kelasSlot:

            if mapel not in mapelGuru:

                mapelGuru[mapel] = guru

            else:

                if mapelGuru[mapel] != guru:

                    pelanggaran += 1

    return pelanggaran

In [574]:
def evaluasiIndividu(individu):

    pelanggaran = 0

    pelanggaran += guruBentrok(individu)

    pelanggaran += distribusiMapel(individu)

    pelanggaran += mapelSiang(individu)

    pelanggaran += durasiGuru(individu)

    pelanggaran += waktuMGMP(individu)

    pelanggaran += cekWaliKelas(individu)

    pelanggaran += konsistensiGuruMapel(individu)

    return pelanggaran * VIOLATION_COST

### Run

In [575]:
def evaluasiPopulasi(populasi):

    fitness = []

    for individu in populasi:

        nilai = evaluasiIndividu(individu)

        fitness.append(nilai)

    return fitness

In [576]:
fitness = evaluasiPopulasi(populasi)

print(min(fitness))

34900


# HELP

In [577]:
def repairDistribusiMapel(individu):

    individu = individu.copy()

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for hari in range(1, 6):

            awal = start + slotAwalHari[hari]
            akhir = start + slotAkhirHari[hari]

            slotHari = individu[awal:akhir]

            kelompok = defaultdict(list)

            for mapel, guru in slotHari:
                kelompok[mapel].append((mapel, guru))

            hasil = []

            for mapel in kelompok:
                hasil.extend(kelompok[mapel])

            # pastikan panjang tetap sama
            if len(hasil) == len(slotHari):
                individu[awal:akhir] = hasil

    return individu

In [578]:
def repairMGMP(individu):

    individu = individu.copy()

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for slot in range(SLOT_PER_KELAS):

            index = start + slot

            mapel, guru = individu[index]

            if mapel not in mgmpMapel:
                continue

            hari = slotKeHari[slot]

            if hari == mgmpMapel[mapel]:

                for s2 in range(SLOT_PER_KELAS):

                    hari2 = slotKeHari[s2]

                    if hari2 != mgmpMapel[mapel]:

                        i2 = start + s2

                        individu[index], individu[i2] = individu[i2], individu[index]
                        break

    return individu

In [579]:
def repairGuruBentrok(individu):

    individu = individu.copy()

    for slot in range(SLOT_PER_KELAS):

        guruMengajar = {}

        for kelas in range(JUMLAH_KELAS):

            index = kelas * SLOT_PER_KELAS + slot

            mapel, guru = individu[index]

            if guru not in guruMengajar:

                guruMengajar[guru] = index

            else:

                start = kelas * SLOT_PER_KELAS

                for s2 in range(SLOT_PER_KELAS):

                    i2 = start + s2

                    mapel2, guru2 = individu[i2]

                    konflik = False

                    for k in range(JUMLAH_KELAS):

                        cek = k * SLOT_PER_KELAS + s2

                        if individu[cek][1] == guru:
                            konflik = True
                            break

                    if not konflik:

                        individu[index], individu[i2] = individu[i2], individu[index]
                        break

    return individu

In [580]:
def repairMapelSiang(individu):

    TARGET_MAPEL = 8

    individu = individu.copy()

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for slot in range(SLOT_PER_KELAS):

            index = start + slot

            mapel, guru = individu[index]

            if mapel != TARGET_MAPEL:
                continue

            hari = slotKeHari[slot]

            batas = batasSiang[hari]

            slotHari = slot - slotAwalHari[hari]

            if slotHari >= batas:

                for s2 in range(SLOT_PER_KELAS):

                    hari2 = slotKeHari[s2]

                    batas2 = batasSiang[hari2]

                    slotHari2 = s2 - slotAwalHari[hari2]

                    if slotHari2 < batas2:

                        i2 = start + s2

                        individu[index], individu[i2] = individu[i2], individu[index]
                        break

    return individu

In [581]:
def updateWolf(wolf, alpha, beta, delta):

    newWolf = wolf.copy()

    for i in range(len(wolf)):

        r = random.random()

        if r < 0.3:

            newWolf[i] = alpha[i]

        elif r < 0.6:

            newWolf[i] = beta[i]

        elif r < 0.8:

            newWolf[i] = delta[i]

        else:

            if random.random() < 0.5:
                newWolf[i] = random.choice([alpha[i], beta[i], delta[i]])

    return newWolf

# GA

In [582]:
def tournamentSelection(populasi, fitness):

    kandidat = random.sample(range(len(populasi)), TOURNAMENT_SIZE)

    best = kandidat[0]

    for i in kandidat:

        if fitness[i] < fitness[best]:

            best = i

    return populasi[best]

In [583]:
def classCrossover(parent1, parent2):

    child = []

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS
        end = start + SLOT_PER_KELAS

        if random.random() < 0.5:

            child.extend(parent1[start:end])

        else:

            child.extend(parent2[start:end])

    return child

In [584]:
def swapMutation(individu):

    individu = individu.copy()

    kelas = random.randint(0, JUMLAH_KELAS - 1)

    start = kelas * SLOT_PER_KELAS

    jumlahSwap = random.randint(3, 6)

    for _ in range(jumlahSwap):

        s1 = random.randint(0, SLOT_PER_KELAS - 1)
        s2 = random.randint(0, SLOT_PER_KELAS - 1)

        i1 = start + s1
        i2 = start + s2

        individu[i1], individu[i2] = individu[i2], individu[i1]

    return individu

In [585]:
def elitism(populasi, fitness):

    pasangan = list(zip(populasi, fitness))

    pasangan.sort(key=lambda x: x[1])

    elite = [ind for ind,fit in pasangan[:ELITE_SIZE]]

    return elite

In [586]:
def gaPhase(populasi, fitness):

    elite = elitism(populasi, fitness)

    newPop = elite.copy()

    while len(newPop) < len(populasi):

        parent1 = tournamentSelection(populasi, fitness)
        parent2 = tournamentSelection(populasi, fitness)

        child = classCrossover(parent1, parent2)

        if random.random() < MUTATION_PROB:
            child = swapMutation(child)

        newPop.append(child)

    return newPop

In [587]:
def gradualRepair(populasi):

    repaired = []

    for individu in populasi:

        individu = repairGuruBentrok(individu)

        individu = repairMapelSiang(individu)

        individu = repairMGMP(individu)

        individu = repairDistribusiMapel(individu)

        repaired.append(individu)

    return repaired

In [588]:
def gwoPhase(populasi, fitness):

    wolves = list(zip(populasi, fitness))

    wolves.sort(key=lambda x: x[1])

    alpha = wolves[0][0]
    beta  = wolves[1][0]
    delta = wolves[2][0]

    newPop = [alpha, beta, delta]

    for i in range(3, len(wolves)):

        wolf = wolves[i][0]

        wolf = updateWolf(wolf, alpha, beta, delta)

        newPop.append(wolf)

    return newPop

In [589]:
populasi = generatePopulation(POPULASI)

fitness = evaluasiPopulasi(populasi)

bestFitnessGlobal = min(fitness)
bestIndividuGlobal = populasi[fitness.index(bestFitnessGlobal)]

for iterasi in range(ITERATION):

    populasi = gaPhase(populasi, fitness)

    populasi = gradualRepair(populasi)

    fitness = evaluasiPopulasi(populasi)

    populasi = gwoPhase(populasi, fitness)

    fitness = evaluasiPopulasi(populasi)

    bestFitnessIter = min(fitness)

    if bestFitnessIter < bestFitnessGlobal:

        bestFitnessGlobal = bestFitnessIter
        bestIndividuGlobal = populasi[fitness.index(bestFitnessIter)]

    print("iterasi", iterasi, "best", bestFitnessGlobal)

iterasi 0 best 14300
iterasi 1 best 10800
iterasi 2 best 9500
iterasi 3 best 8000
iterasi 4 best 7300
iterasi 5 best 6300
iterasi 6 best 6100
iterasi 7 best 6100
iterasi 8 best 5600
iterasi 9 best 5500
iterasi 10 best 5100
iterasi 11 best 5100
iterasi 12 best 4600
iterasi 13 best 4600
iterasi 14 best 4600
iterasi 15 best 4600
iterasi 16 best 4600
iterasi 17 best 4600
iterasi 18 best 4600
iterasi 19 best 4600
iterasi 20 best 4600
iterasi 21 best 4600
iterasi 22 best 4600
iterasi 23 best 4600
iterasi 24 best 4600
iterasi 25 best 4600
iterasi 26 best 4600
iterasi 27 best 4600
iterasi 28 best 4400
iterasi 29 best 4400
iterasi 30 best 4400
iterasi 31 best 4400
iterasi 32 best 4300
iterasi 33 best 4300
iterasi 34 best 4300
iterasi 35 best 4300
iterasi 36 best 4300
iterasi 37 best 4300
iterasi 38 best 4300
iterasi 39 best 4300
iterasi 40 best 4300
iterasi 41 best 4300
iterasi 42 best 4300
iterasi 43 best 4200
iterasi 44 best 4200
iterasi 45 best 4200
iterasi 46 best 4200
iterasi 47 best 4200


# DEBUG

In [590]:
def evaluasiDetail(individu):

    g1 = guruBentrok(individu)
    g2 = distribusiMapel(individu)
    g3 = mapelSiang(individu)
    g4 = durasiGuru(individu)
    g5 = waktuMGMP(individu)
    g6 = cekWaliKelas(individu)
    g7 = konsistensiGuruMapel(individu)

    print(
        "guruBentrok:", g1,
        "distribusi:", g2,
        "siang:", g3,
        "durasi:", g4,
        "mgmp:", g5,
        "wali:", g6,
        "konsistensi:", g7
    )

In [591]:
evaluasiDetail(populasi[1])

guruBentrok: 12 distribusi: 0 siang: 0 durasi: 0 mgmp: 0 wali: 13 konsistensi: 0


In [592]:
populasi[1][:36]

[(10, 15),
 (10, 15),
 (11, 43),
 (11, 43),
 (3, 36),
 (3, 36),
 (3, 36),
 (6, 44),
 (8, 17),
 (6, 44),
 (1, 31),
 (1, 31),
 (7, 37),
 (7, 37),
 (4, 19),
 (5, 48),
 (5, 48),
 (5, 48),
 (5, 48),
 (8, 17),
 (13, 53),
 (13, 53),
 (4, 19),
 (3, 36),
 (9, 38),
 (9, 38),
 (12, 55),
 (11, 43),
 (11, 43),
 (2, 39),
 (6, 44),
 (2, 39),
 (4, 19),
 (4, 19),
 (3, 36),
 (10, 15)]